# Notebook 0 — Diffusion from Scratch in 1D

> **Reference:** Ho, Jain & Abbeel — *Denoising Diffusion Probabilistic Models* (NeurIPS 2020)

## Generative modeling = learning a distribution

> A diffusion model learns the data distribution $p(x)$ by *destroying* structure with noise (forward $q$) and *learning to reverse* it ($p_\theta$). On the slides $p(x)$ is abstract; here it is a 1-D bimodal you can literally **draw** — so every claim becomes something you can see.

A generative model learns the **probability distribution** of some data so it can produce new, plausible samples. This is either:
- **Explicit:** the model learns $p(x)$ directly (VAEs, normalizing flows, **diffusion models**).
- **Implicit:** the model learns to produce realistic samples without ever computing $p(x)$ (GANs).

On images this distribution lives in thousands of dimensions and we can never *see* it, we only judge samples by eye. So here we shrink the problem to **1D**: our data is a mixture of two Gaussians, one at $-3$ and one at $+3$. 

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

seed = 123
torch.manual_seed(seed); np.random.seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## The data: a 1D bimodal distribution

Our "dataset" is $\;0.5\,\mathcal N(-3, 1) + 0.5\,\mathcal N(+3, 1)$: flip a coin, then draw from the left or the right bump.

In Notebook 1 each datapoint is a $16\times16\times3$ sprite — a point in 768-dimensional space. Here each datapoint is **one number**, so the whole "dataset" is something we can plot in full.

In [ ]:
MU, SD = 3.0, 1.0
def sample_bimodal(n):
    center = torch.where(torch.rand(n, device=device) < 0.5, -MU, MU)   # pick mode -3 or +3
    return center + SD * torch.randn(n, device=device)

data = sample_bimodal(5000)
plt.figure(figsize=(7, 3))
plt.hist(data.cpu().numpy(), bins=80, density=True, alpha=0.8)
plt.axvline(-3, color="k", ls=":", alpha=.4); plt.axvline(3, color="k", ls=":", alpha=.4)
plt.title("Our data: a mixture of two Gaussians at -3 and +3"); plt.xlabel("x")
plt.tight_layout(); plt.show()

## 1. The Forward Process

> **_Forward Process_.** Same equations as the deck: the one-step kernel $q(x_t\mid x_{t-1})=\mathcal N(\sqrt{1-\beta_t}\,x_{t-1},\,\beta_t)$ and the closed form $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon$. The system is driven to its equilibrium $\mathcal N(0,1)$ :**the bimodal "equilibrate" into a single Gaussian.**

We gradually corrupt a datapoint $x_0$ by adding Gaussian noise over $T$ steps. One step:
$$x_t = \sqrt{1-\beta_t}\,x_{t-1} + \sqrt{\beta_t}\,\varepsilon, \qquad \varepsilon\sim\mathcal N(0,1),$$
and the **closed-form shortcut** lets us jump straight to any $t$:
$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon, \qquad \bar\alpha_t = \prod_{i\le t}(1-\beta_i).$$

We use **linear** schedule ($\beta_t$ from $10^{-4}$ to $0.02$). After enough steps the data is destroyed into pure $\mathcal N(0,1)$, and the bimodal structure disappears.

In [ ]:
T_STEPS = 1000
betas = torch.linspace(1e-4, 0.02, T_STEPS).to(device)   # same linear schedule as Notebook 1
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def forward_sample(x0, t, noise=None):
    """Closed-form corruption: x_t = sqrt(ab) x0 + sqrt(1-ab) noise.  (same signature as NB1)"""
    if noise is None:
        noise = torch.randn_like(x0)
    ab = alpha_bar[t]
    return torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise, noise

# Watch the whole distribution get destroyed as t grows
x0 = sample_bimodal(5000)
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, ts in zip(axes, [0, 100, 400, 999]):
    t = torch.full((x0.shape[0],), ts, device=device, dtype=torch.long)
    xt, _ = forward_sample(x0, t)
    ax.hist(xt.cpu().numpy(), bins=70, density=True, alpha=0.8)
    ax.set_title(f"t = {ts}   (ab = {alpha_bar[ts].item():.3f})")
plt.suptitle("Forward process: the bimodal data dissolves into N(0,1)")
plt.tight_layout(); plt.show()

## 2. Training

> **_Reverse Process_ (predicting the noise)** & _ELBO_ → $\mathcal L_{\text{simple}}$. We predict $\varepsilon$, invert to $\hat x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\,\varepsilon}{\sqrt{\bar\alpha_t}}$, and show through the ELBO and its KL term $L_{t-1}$, that the whole objective collapses to one line: $\mathcal L_{\text{simple}}=\mathbb E\,\lVert\varepsilon-\varepsilon_\theta(x_t,t)\rVert^2$. **Below we train exactly that loss, then check the network really learned $\mathbb E[\varepsilon\mid x_t,t]$.**

We want to **invert** the corruption. From the closed-form formula, if we can predict the noise $\varepsilon$ that was added to get to $x_t$, we can recover $x_0$:
$$\hat x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\varepsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}.$$

So we train a network $\varepsilon_\theta(x_t, t)$ to predict the noise, minimizing the simple DDPM objective (Ho et al. 2020):
$$\mathcal L = \mathbb E_{t,\,x_0,\,\varepsilon}\big[\,\lVert \varepsilon - \varepsilon_\theta(x_t, t)\rVert^2\,\big].$$

**What does "predict the noise" really mean?** Many different $(x_0,\varepsilon)$ produce the *same* $x_t$ so the information is genuinely lost. So the best the network can do, minimizing MSE, is output the **average** noise consistent with that $x_t$:
$$\varepsilon_\theta(x_t,t)\;\longrightarrow\;\mathbb E[\varepsilon\mid x_t,t].$$

In the followin the denoiser is a U-Net. Here it is the smallest thing that could work: an MLP that takes $x_t$ and an embedding of $t$, and outputs one number.

In [ ]:
def t_embedding(t, dim=64):
    """Sinusoidal time embedding (same idea as NB1): integer step -> continuous vector."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    a = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.sin(a), torch.cos(a)], dim=-1)

class EpsMLP(nn.Module):
    """The 1D denoiser: (x_t, t) -> predicted noise. NB1's U-Net, shrunk to a few Linear layers."""
    def __init__(self, t_dim=64, h=128):
        super().__init__()
        self.t_dim = t_dim
        self.net = nn.Sequential(
            nn.Linear(1 + t_dim, h), nn.SiLU(),
            nn.Linear(h, h), nn.SiLU(),
            nn.Linear(h, h), nn.SiLU(),
            nn.Linear(h, 1),
        )
    def forward(self, x, t):                     # x: (B,)  t: (B,) integers
        h = torch.cat([x[:, None], t_embedding(t, self.t_dim)], dim=-1)
        return self.net(h).squeeze(-1)           # (B,)

model = EpsMLP().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}  (the U-Net in NB1 has ~16,000,000)")

### The training loop

The exact same loop as Notebook 1 — only the tensor shapes differ:
```
x0       = a batch of real data        # sample from the bimodal
t        = random noise levels         # one per example
eps      = real noise ~ N(0,1)
x_t      = forward_sample(x0, t, eps)  # corrupt
eps_pred = model(x_t, t)               # predict the noise
loss     = mean((eps - eps_pred)**2)   # MSE
loss.backward(); opt.step()
```

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
for step in range(6000):
    x0 = sample_bimodal(256)
    x0 = torch.cat([x0, -x0])                     # our 1D data is symmetric (modes at -3 and +3):
                                                  # training on each point AND its mirror keeps the learned
                                                  # eps field symmetric, so DDPM reconstructs both modes
                                                  # equally (otherwise a tiny RNG-driven bias makes one
                                                  # lobe weaker). A free luxury of knowing p(x) is symmetric.
    t  = torch.randint(0, T_STEPS, (x0.shape[0],), device=device)
    x_t, eps = forward_sample(x0, t)              # corrupt with fresh noise
    eps_pred = model(x_t, t)                      # predict it
    loss = ((eps_pred - eps) ** 2).mean()         # MSE on the noise
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

plt.figure(figsize=(7, 3))
plt.plot(losses, alpha=0.35)
plt.plot(np.convolve(losses, np.ones(100) / 100, "valid"))
plt.xlabel("step"); plt.ylabel("MSE"); plt.title(f"Training loss (final ~ {np.mean(losses[-200:]):.3f})")
plt.tight_layout(); plt.show()

### Did it learn the right thing? Compare against the oracle

The loss starts near $1$ — which is exactly the variance of $\varepsilon$: a network that always predicts $0$ scores MSE $=1$. **Every bit below 1 is noise structure the network has learned to explain.**

But *what* did it learn? In 1D we can cheat: for a mixture of Gaussians the true answer $\mathbb E[\varepsilon\mid x_t,t]$ has a **closed form**. We compute it and overlay it on the network's output. If they match, the network really has learned the conditional mean of the noise.

In [ ]:
@torch.no_grad()
def eps_oracle(x_t, t):
    """Exact E[eps | x_t, t] for 0.5 N(-MU,SD^2) + 0.5 N(MU,SD^2)."""
    ab = alpha_bar[t]; sq = torch.sqrt(ab); s1 = torch.sqrt(1 - ab)
    means = torch.tensor([-MU, MU], device=x_t.device)
    var_t = ab * SD**2 + (1 - ab)
    logw = -0.5 * (x_t[:, None] - sq * means[None, :])**2 / var_t      # posterior weight of each mode
    w = torch.softmax(logw, dim=1)
    post_var = 1.0 / (1.0 / SD**2 + ab / (1 - ab))
    post_mean = post_var * (means[None, :] / SD**2 + sq * x_t[:, None] / (1 - ab))
    x0_hat = (w * post_mean).sum(1)                                    # E[x0 | x_t]
    return (x_t - sq * x0_hat) / s1                                    # -> E[eps | x_t]

model.eval()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, ts in zip(axes, [100, 500, 900]):
    xs = torch.linspace(-8, 8, 400, device=device)
    t = torch.full((400,), ts, device=device, dtype=torch.long)
    with torch.no_grad():
        ep = model(xs, t)
    ax.plot(xs.cpu(), eps_oracle(xs, ts).cpu(), "k--", lw=2, label="oracle  E[eps|x_t]")
    ax.plot(xs.cpu(), ep.cpu(), "C3", lw=2, label="network  eps_theta")
    ax.set_title(f"t = {ts}"); ax.set_xlabel("x_t"); ax.set_ylabel("predicted eps"); ax.legend(fontsize=8)
plt.suptitle("The network learned the conditional mean of the noise (= the oracle)")
plt.tight_layout(); plt.show()

The two curves match wherever $x_t$ has support. Look at $t=100$: near $x_t=0$ there is a **step**. There the "correct" noise flips sharply, because a slightly negative $x_t$ almost certainly came from the $-3$ mode and a slightly positive one from the $+3$ mode — the network learned *that decision* too. At large $t$ the relation is nearly linear (with lots of noise $x_t\approx\sqrt{1-\bar\alpha_t}\,\varepsilon$, so $\mathbb E[\varepsilon\mid x_t]\approx x_t/\sqrt{1-\bar\alpha_t}$).

This is the whole game: **"learning to reconstruct the noise" = learning $\mathbb E[\varepsilon\mid x_t,t]$**, the conditional mean.

### The learned noise field, everywhere

The oracle check above compared the network to the truth at three time slices. Here is the same prediction over the *whole* plane. For every position $x_t$ (horizontal) and every noise level $t$ (vertical), the color is the predicted noise $\hat\varepsilon_\theta(x_t,t)$: red and blue are opposite signs, white is near zero. The data histogram sits on top, and the dashed lines mark the two modes.

Read it from top (little noise) to bottom (almost pure noise). At low $t$ the prediction flips sharply right at the midpoint between the modes, the same step we saw in the oracle plot. At high $t$ it smooths into an almost linear ramp, because there the position barely tells you which mode the point came from.

In [ ]:
# The network's predicted noise over the whole (x_t, t) plane.
t_samples = np.linspace(0, T_STEPS - 1, 100).astype(int)
x_samples = np.linspace(-8, 8, 200)
x_min, x_max = x_samples[0], x_samples[-1]

model.eval()
eps_image = np.zeros((len(t_samples), len(x_samples)))
for i, t in enumerate(t_samples):
    x_t = torch.tensor(x_samples, dtype=torch.float32, device=device)
    t_tensor = torch.full((len(x_t),), int(t), device=device, dtype=torch.long)
    with torch.no_grad():
        eps_image[i] = model(x_t, t_tensor).cpu().numpy()

x0_vis = sample_bimodal(20000).cpu().numpy()

fig = plt.figure(figsize=(9, 7))
gs = fig.add_gridspec(2, 2, width_ratios=[20, 1], height_ratios=[1, 4], hspace=0.08, wspace=0.05)
ax_hist = fig.add_subplot(gs[0, 0])
ax_heat = fig.add_subplot(gs[1, 0], sharex=ax_hist)
cax = fig.add_subplot(gs[1, 1])

ax_hist.hist(x0_vis, bins=np.linspace(x_min, x_max, 100), density=True, alpha=0.75, color="orange")
ax_hist.set_xlim(x_min, x_max)
ax_hist.set_ylabel(r"$p_{data}(x_0)$")
ax_hist.set_title(r"Data distribution and the network's predicted noise $\hat{\varepsilon}_\theta(x_t,t)$")
ax_hist.tick_params(labelbottom=False)

im = ax_heat.imshow(eps_image, extent=[x_min, x_max, t_samples[-1], t_samples[0]],
                    aspect="auto", cmap="RdBu_r")
ax_heat.set_xlim(x_min, x_max)
ax_heat.set_xlabel(r"$x_t$"); ax_heat.set_ylabel("t")
ax_heat.axvline(-MU, color="orange", ls="--", lw=1, alpha=0.7)
ax_heat.axvline(MU, color="orange", ls="--", lw=1, alpha=0.7)
fig.colorbar(im, cax=cax, label=r"$\hat{\varepsilon}_\theta(x_t,t)$")
plt.show()

## 3. Sampling

> **_Reverse Process_ (generative model).** The tractable posterior $q(x_{t-1}\mid x_t,x_0)=\mathcal N(\tilde\mu_t,\tilde\beta_t)$ can be sampled $x_{t-1}=\tilde\mu_t+\sqrt{\tilde\beta_t}\,z$, iterating $t:T\to1$. A *single* $\hat x_0$ cannot represent the full uncertainty. **Below you see exactly that:** one jump collapses to the average; the stochastic iteration recovers both modes.

To generate we start from pure noise $x_T\sim\mathcal N(0,1)$ and **reverse** the corruption with the trained network. But how big a step can we take?

**One jump is not enough.** The network only ever returns the *average* $x_0$ consistent with a given $x_t$ — that is exactly what MSE training buys us (section 2). Halfway between the two modes that average is the empty middle. We can see it directly: corrupt real samples to a high noise level, then predict $\hat x_0$ in a single step — everything collapses toward $0$, and the bimodality is gone.

**DDPM takes many small steps instead.** At each step we predict the noise, step back a little, and re-inject a bit of fresh noise so the trajectory can *commit* to one mode:
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t,t)\Big) + \sigma_t z,\qquad \sigma_t=\sqrt{\tilde\beta_t}.$$
Starting from pure noise, it recovers both peaks.

In [ ]:
# One jump: corrupt real data to a high noise level, then ask for x0 in a single step.
# It returns E[x0 | x_t] -- the average -- which between the modes sits near 0.
T_ONESHOT = 700
@torch.no_grad()
def one_shot_x0(n_samples=4000, t_demo=T_ONESHOT):
    x0 = sample_bimodal(n_samples)
    t  = torch.full((n_samples,), t_demo, device=device, dtype=torch.long)
    x_t, _ = forward_sample(x0, t)
    ab = alpha_bar[t_demo]
    eps = model(x_t, t)
    return (x_t - torch.sqrt(1 - ab) * eps) / torch.sqrt(ab)

# Many steps: the full reverse process, starting from pure noise.
@torch.no_grad()
def ddpm_sample(model, n_samples=4000, t_steps=T_STEPS):
    x = torch.randn(n_samples, device=device)            # x_T ~ N(0,1)
    for t in reversed(range(t_steps)):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        eps = model(x, t_batch)
        a_t, ab_t, b_t = alphas[t], alpha_bar[t], betas[t]
        mean = (1 / torch.sqrt(a_t)) * (x - (b_t / torch.sqrt(1 - ab_t)) * eps)
        if t > 0:
            beta_tilde = ((1 - alpha_bar[t - 1]) / (1 - ab_t)) * b_t
            x = mean + torch.sqrt(beta_tilde) * torch.randn(n_samples, device=device)
        else:
            x = mean                                      # last step: no noise
    return x

real     = sample_bimodal(4000)
one_shot = one_shot_x0()
multi    = ddpm_sample(model)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True, sharey=True)
bins = np.linspace(-7, 7, 70)
axes[0].hist(real.cpu().numpy(),     bins=bins, density=True, alpha=.8);             axes[0].set_title("real data")
axes[1].hist(one_shot.cpu().numpy(), bins=bins, density=True, alpha=.8, color="C1"); axes[1].set_title(f"one shot from t={T_ONESHOT} (averages)")
axes[2].hist(multi.cpu().numpy(),    bins=bins, density=True, alpha=.8, color="C2"); axes[2].set_title("DDPM (1000 steps)")
for ax in axes:
    ax.axvline(-3, color="k", ls=":", alpha=.3); ax.axvline(3, color="k", ls=":", alpha=.3)
plt.suptitle("One jump returns the average (bimodality lost); iterating recovers both modes")
plt.tight_layout(); plt.show()

print(f"generated (DDPM): mean={multi.mean().item():.2f}, std={multi.std().item():.2f}  |  "
      f"real: mean={real.mean().item():.2f}, std={real.std().item():.2f}")

### Sampling as flowing down a field

Sampling just means applying the learned reverse step again and again. The gray arrows show the average reverse move $(\mu_\theta(x_t,t)-x_t,\,-1)$ across the $(x,t)$ plane: at each height $t$, which way the mean nudges a sample as $t$ steps down. The colored curves are a few stochastic runs, each from $x_T\sim\mathcal N(0,1)$ at the top to a generated $x_0$ at the bottom.

The bottom panel compares the data with the full generated distribution (DDPM, many samples): both peaks come back, and they are symmetric. Each colored path is just a single draw, so some end on a mode and some in the lower-density region between them, but the bulk lands on the two peaks.

In [ ]:
# Sampling = repeatedly following the learned reverse step.
@torch.no_grad()
def reverse_mean(x, t):
    """DDPM reverse mean mu_theta(x_t, t) and the predicted noise (same formula as ddpm_sample)."""
    t_tensor = torch.full(x.shape, int(t), device=x.device, dtype=torch.long)
    eps = model(x, t_tensor)
    a_t, ab_t, b_t = alphas[t], alpha_bar[t], betas[t]
    mu = (1 / torch.sqrt(a_t)) * (x - (b_t / torch.sqrt(1 - ab_t)) * eps)
    return mu, eps

@torch.no_grad()
def sample_trajectory(x_start=None):
    """One reverse run from pure noise, recording (x, t) at every step."""
    x = torch.randn(1, device=device) if x_start is None else torch.tensor([x_start], device=device)
    xs, ts = [x.item()], [T_STEPS - 1]
    for t in reversed(range(1, T_STEPS)):
        mu, _ = reverse_mean(x, t)
        beta_tilde = ((1 - alpha_bar[t - 1]) / (1 - alpha_bar[t])) * betas[t]
        x = mu + torch.sqrt(beta_tilde) * torch.randn(1, device=device)
        xs.append(x.item()); ts.append(t - 1)
    return np.array(xs), np.array(ts)

# a few stochastic reverse trajectories (paths), plus the full generated distribution
# torch.manual_seed(12)
trajs = [sample_trajectory() for _ in range(2)]
generated = ddpm_sample(model, n_samples=4000).cpu().numpy()   # many samples: recovers both modes

# the average reverse field on a grid: (x_t, t) -> (mu_theta - x_t, -1)
t_grid = np.linspace(T_STEPS - 1, 1, 24).astype(int)
x_grid = np.linspace(-8, 8, 41)
X, TT = np.meshgrid(x_grid, t_grid)
U = np.zeros_like(X, dtype=float)
for i, t in enumerate(t_grid):
    x_vals = torch.tensor(x_grid, dtype=torch.float32, device=device)
    mu, _ = reverse_mean(x_vals, int(t))
    U[i, :] = (mu - x_vals).cpu().numpy()

# scale arrows only for readability: we show the direction of the mean field
arrow_len = 22.0
norm = np.sqrt(U**2 + 1.0) + 1e-8
U_plot = arrow_len * U / norm
V_plot = -arrow_len * 1.0 / norm

colors = ["C3", "C0", "C2", "C4", "C5", "C1"]

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": [4, 1]})
ax = axes[0]
ax.quiver(X, TT, U_plot, V_plot, angles="xy", scale_units="xy", scale=1,
          width=0.0032, headwidth=4.5, headlength=5.5, headaxislength=4.5, alpha=0.5, color="gray")
for (xs, ts), col in zip(trajs, colors):
    ax.plot(xs, ts, color=col, lw=2.0, alpha=0.9)
    ax.scatter(xs[0], ts[0], color="black", s=24, zorder=5)
    ax.scatter(xs[-1], ts[-1], color=col, s=42, zorder=5)
ax.plot([], [], color="gray", lw=2.0, label="reverse trajectories")
ax.scatter([], [], color="black", s=24, label=r"$x_T$ start")
ax.axvline(-MU, color="orange", ls="--", lw=1.2, alpha=0.7)
ax.axvline(MU, color="orange", ls="--", lw=1.2, alpha=0.7)
ax.set_ylabel("t"); ax.set_title("1D denoising trajectories and the learned mean reverse field")
ax.legend(loc="upper right"); ax.grid(alpha=0.15)
ax.set_xlim(-8, 8); ax.set_ylim(T_STEPS - 1, 0)   # noise at top, data at bottom

# bottom: the full generated distribution vs the data -> both peaks reconstructed
bins = np.linspace(-8, 8, 100)
axes[1].hist(sample_bimodal(20000).cpu().numpy(), bins=bins, density=True, alpha=0.55, color="orange", label="data")
axes[1].hist(generated, bins=bins, density=True, histtype="step", color="black", lw=1.8, label="generated (DDPM)")
axes[1].axvline(-MU, color="orange", ls="--", lw=1.0, alpha=0.6); axes[1].axvline(MU, color="orange", ls="--", lw=1.0, alpha=0.6)
axes[1].set_xlabel(r"$x$"); axes[1].set_ylabel(r"$p_{data}$"); axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

print("trajectory end points:", ", ".join(f"{xs[-1]:+.2f}" for xs, _ in trajs))